### IMMUNOtron pipeline

Data were generated using the **IMMUNOtron robotic pipeline** (Achar et al., *Science*, 2022; doi:10.1126/science.abl5311).

Cytokine dynamics analysis was adapted from code developed by **Grégoire Altan-Bonnet and colleagues at the National Cancer Institute (NCI)**.

# 1/ Data import

In [ ]:
#%% Imports

import os
import pickle as pkl

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

from numpy import trapz
from scipy import integrate
from sklearn import metrics
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split



#%% Paths

# Repository structure:
# CART_immuno_metabolic_pipeline/
# ├── data/
# └── code/

file_path = "../data/cytokineConcentrationPickleFile-20221003-MetMYCAR_1.pkl"

# Load data
data = pd.read_pickle(file_path)

# Extract experiment name from filename
filename = os.path.basename(file_path)
Experiment = filename[filename.find("-") + 1 : filename.find(".pkl")]
print("Experiment:", Experiment)


#%% Reprocess cytokine dataset

# Optional: remove one time point if needed
time_point_to_remove = None  # e.g. 48.0

if time_point_to_remove is not None and time_point_to_remove in data.columns:
    data = data.drop(columns=[time_point_to_remove])

ExperimentalVariables = list(data.index.names)

data = data.stack()
data = data.reset_index()
data = data.rename(columns={data.columns[-2]: "Time"})
data.set_index(ExperimentalVariables + ["Time"], inplace=True)

data = data.unstack(level="Cytokine")
data.columns = [x[1] for x in data.columns]

ExperimentalVariables.remove("Cytokine")
Cytokines = data.columns

df4 = data.stack()

temp = list(df4.index.names)
temp[-1] = "Cytokine"
df4.index.names = temp

df4 = df4.to_frame().reset_index()
df4 = df4.rename(columns={0: "Concentration"})

display(df4.head())
print(df4.columns)
print(df4.shape)

# 2/ Raw cytokine/chemokines curves

In [ ]:
#%% Plot cytokine responses

plottingDf = df4.copy()

custom_palette = {
    "Mock": "gray",
    "19_BBz": "red",
    "19_28z": "darkred",
    "22_BBz": "dodgerblue",
    "22_28z": "darkblue",
    "33_BBz": "mediumseagreen",
    "33_28z": "darkgreen",
    "None": "black",
}

g = sns.relplot(
    data=plottingDf,
    x="Time",
    y="Concentration",
    kind="line",
    marker="o",
    err_style="bars",
    err_kws={"capsize": 4},

    hue="CAR",
    style="EffectorTargetRatio",

    row=ExperimentalVariables[2],
    col="Cytokine",

    palette=custom_palette,

    facet_kws={
        "margin_titles": True,
        "sharey": "row",
        "sharex": True,
    },
)

# Scales
g.set(yscale="log")
g.set(xscale="linear")

# Title
g.figure.suptitle(f"Cytokine responses - {Experiment}", fontsize=24)

# Layout
g.figure.tight_layout(rect=[0, 0.03, 0.9, 0.95])

# Save figure
save_dir = "../output"
os.makedirs(save_dir, exist_ok=True)

g.savefig(
    os.path.join(save_dir, f"log_cytokine_{Experiment}.pdf"),
    bbox_inches="tight",
)

# 3/ Log-average cytokine concentrations

In [ ]:
#%%
Index_5=ExperimentalVariables+['Cytokine']
df5=df4.set_index(Index_5)
Conditions=df5.index.unique()

df6=pd.DataFrame(np.zeros(len(Conditions)),index=Conditions,columns=['Log Average'])
for cc in Conditions:
    logaverage=np.mean(np.log10(df5['Concentration'].loc[cc]/1e-3))
    df6.loc[cc]=[np.power(10,logaverage)]
df6=df6.unstack(['Cytokine'])

# 4/ Log-cytokine integrals over time
### ∫(t₀ → tₓ)

## Compute ∫(t₀ → tₓ) Log[Cytokine]


In [ ]:
#%% Compute integral of log[Cytokine] as a function of time

Index_5 = list(df4.columns)
Index_5.remove("Concentration")
Index_5[-2], Index_5[-1] = Index_5[-1], Index_5[-2]

df5 = df4.set_index(Index_5)
df5 = df5.sort_index()

# Log10-transform cytokine concentrations
df5 = np.log10(df5 / 1e-3)
df5 = df5.dropna()
df5.reset_index(inplace=True, level=["Time"])

LogIntegral_df = df5.copy()
LogIntegral_df.rename(
    columns={"Concentration": "Integral(Log)"},
    inplace=True,
)

Conditions = df5.index.unique()

for cc in Conditions:
    y = df5.loc[cc]["Concentration"].values
    time = df5.loc[cc]["Time"].values

    y_int = integrate.cumulative_trapezoid(
        y,
        time,
        initial=0,
    )
    y_int[0] = y[0]

    LogIntegral_df.loc[cc] = np.stack(
        [time, y_int],
        axis=1,
    )

LogIntegral_df.reset_index(inplace=True)
LogIntegral_df.set_index(
    list(LogIntegral_df.columns[:-1]),
    inplace=True,
)
LogIntegral_df = LogIntegral_df.sort_index()
LogIntegral_df = LogIntegral_df.unstack(level=["Cytokine"])

## PCA

In [ ]:
#%%
LogIntegral_df=LogIntegral_df.dropna()
X=LogIntegral_df.values
pca = PCA(n_components=3)
X_new=pca.fit_transform(X,y=None)
PCA_LogAverageOverTime_df=pd.DataFrame(X_new,index=LogIntegral_df.index,\
                 columns=['PCA1','PCA2','PCA3'])
loadings_LogAverageOverTime= pd.DataFrame(pca.components_.T,\
                                    columns=['PC1_loadings', 'PC2_loadings','PC3_loadings'],\
                                    index=LogIntegral_df.columns)
PCA_LogAverageOverTime_df.reset_index(inplace=True)

### Analysis of PCA components

In [ ]:
# PCA loadings plot

fig_width = 1.75
fig_height = 2

main_title = ""
subtitle = Experiment


# Figure style
mpl.rcParams["font.family"] = "sans-serif"
mpl.rcParams["font.sans-serif"] = ["Liberation Sans"]

mpl.rcParams["font.size"] = 6
mpl.rcParams["axes.titlesize"] = 6
mpl.rcParams["axes.labelsize"] = 6
mpl.rcParams["xtick.labelsize"] = 5
mpl.rcParams["ytick.labelsize"] = 5
mpl.rcParams["legend.fontsize"] = 6

mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42

mpl.rcParams["axes.linewidth"] = 0.5
mpl.rcParams["xtick.major.width"] = 0.5
mpl.rcParams["ytick.major.width"] = 0.5
mpl.rcParams["xtick.major.size"] = 2
mpl.rcParams["ytick.major.size"] = 2


# Data
df = loadings_LogAverageOverTime.reset_index()

ytick_labels = [
    x.replace("IFNg", "IFNγ")
     .replace("TNFa", "TNFα")
    for x in df["Cytokine"].tolist()
]


# Figure
fig, axes = plt.subplots(
    1, 3,
    figsize=(fig_width, fig_height),
    sharey=True
)

grey_color = "0.5"


# Bar plots
sns.barplot(
    data=df,
    y="Cytokine",
    x="PC1_loadings",
    ax=axes[0],
    color=grey_color
)

sns.barplot(
    data=df,
    y="Cytokine",
    x="PC2_loadings",
    ax=axes[1],
    color=grey_color
)

sns.barplot(
    data=df,
    y="Cytokine",
    x="PC3_loadings",
    ax=axes[2],
    color=grey_color
)


# Axes
for i, ax in enumerate(axes):

    ax.set_xlabel(
        f"PC{i+1} – {pca.explained_variance_ratio_[i] * 100:.1f}%",
        fontsize=5
    )

    ax.set_ylabel("")

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax.axvline(
        0,
        color="black",
        linewidth=0.5,
        alpha=0.6
    )

axes[0].set_yticks(range(len(df)))
axes[0].set_yticklabels(ytick_labels)

for ax in axes[1:]:
    ax.tick_params(labelleft=False)


# Optional title
expl_var_text = (
    f"PCA1>{pca.explained_variance_ratio_[0] * 100:.1f}% | "
    f"PCA2>{pca.explained_variance_ratio_[1] * 100:.1f}% | "
    f"PCA3>{pca.explained_variance_ratio_[2] * 100:.1f}%"
)

plt.suptitle(
    f"{main_title}\n{expl_var_text}\n{subtitle}" if main_title else "",
    y=0.98,
    fontsize=7
)

plt.tight_layout(rect=[0, 0, 1, 0.85])


# Save figure
plt.savefig(
    os.path.join(
        save_dir,
        f"Loadings_PCA_Integral_log_OverTime_{Experiment}.pdf"
    ),
    bbox_inches="tight"
)

### Time-resolved PCA trajectories

In [ ]:
# 2D PCA trajectories

for i, j in [(1, 2), (1, 3), (2, 3)]:
    g = sns.relplot(
        data=PCA_LogAverageOverTime_df,
        x="PCA" + str(i),
        y="PCA" + str(j),
        hue=ExperimentalVariables[0],
        style=ExperimentalVariables[1],
        row=ExperimentalVariables[2],
        col=ExperimentalVariables[3],
        palette="viridis",
        kind="line",
        facet_kws={
            "sharey": True,
            "sharex": True
        }
    )

    g.figure.suptitle(
        f"PCA{j} vs PCA{i} of cytokine responses over time\n{Experiment}",
        fontsize=16
    )

    g.figure.tight_layout(rect=[0, 0.03, 0.9, 0.9])

    g.savefig(
        os.path.join(
            save_dir,
            f"PCA{j}vsPCA{i}_timedlog_cytokine_{Experiment}.pdf"
        ),
        bbox_inches="tight"
    )

### Plot 2D PCA (t) trajectories

### All trajectories

In [ ]:
# PCA1 vs PCA2 trajectories

# Global style
mpl.rcParams["font.family"] = "sans-serif"
mpl.rcParams["font.sans-serif"] = ["Liberation Sans"]
mpl.rcParams["font.size"] = 6
mpl.rcParams["axes.titlesize"] = 8
mpl.rcParams["axes.labelsize"] = 6
mpl.rcParams["xtick.labelsize"] = 4
mpl.rcParams["ytick.labelsize"] = 4
mpl.rcParams["legend.fontsize"] = 6
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42

SPINE_W = 0.5
TICK_W_MAJOR = 0.5
TICK_LEN_MAJ = 2
TICK_LABEL_PAD = 0.5

EXPORT_DPI = 600

sns.set_style("white")


# Save path
save_path = os.path.join(
    save_dir,
    "PCA2D_all_trajectories.pdf"
)


# Colors
custom_palette = {
    "Mock": "gray",
    "19_BBz": "red",
    "19_28z": "darkred",
    "22_BBz": "dodgerblue",
    "22_28z": "darkblue",
    "33_BBz": "mediumseagreen",
    "33_28z": "darkgreen",
    "None": "black",
}


# Plot
g = sns.relplot(
    data=PCA_LogAverageOverTime_df,
    x="PCA1",
    y="PCA2",
    hue="CAR",
    style="EffectorTargetRatio",
    units="Replicate",
    estimator=None,
    kind="line",
    palette=custom_palette,
    lw=0.4,
    alpha=0.6,
    height=1.5,
    aspect=1.3,
    legend="full"
)


# Style axes
for ax in g.axes.flat:
    for s in ["left", "bottom"]:
        ax.spines[s].set_linewidth(SPINE_W)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax.tick_params(
        axis="both",
        which="major",
        direction="out",
        length=TICK_LEN_MAJ,
        width=TICK_W_MAJOR,
        pad=TICK_LABEL_PAD
    )

    ax.set_xlabel("PCA1")
    ax.set_ylabel("PCA2")


# Legend
if g._legend is not None:
    g._legend.set_title("CAR / E:T")
    g._legend.set_frame_on(False)

sns.move_legend(
    g,
    "upper left",
    bbox_to_anchor=(0.9, 1.0)
)

g.fig.subplots_adjust(
    left=0.20,
    right=0.75,
    bottom=0.20,
    top=0.95
)

g.fig.canvas.draw()


# Save
g.fig.savefig(
    save_path,
    format="pdf",
    dpi=EXPORT_DPI,
    bbox_inches="tight",
    pad_inches=0.01
)

print("Saved to:", save_path)

plt.show()

# 5/ Angular parametrization of PCA (t) trajectories


### Compute angle -distance

In [ ]:
#%%Compute Angle and Distance from origin for PCA trajectories in 2D

theta_shift = 180*np.pi/180
def compute_angle(x,y,theta_shift,direction='counterclockwise'):
    if direction == "clockwise":
        return 2*np.pi - np.mod(np.arctan2(y, x) - theta_shift, 2*np.pi) / (2*np.pi)
    elif direction == "counterclockwise":
        return np.mod(np.arctan2(y, x) - theta_shift, 2*np.pi)  / (2*np.pi)


initial_timepoint=PCA_LogAverageOverTime_df['Time'].min()
LatentSpace=PCA_LogAverageOverTime_df.set_index(ExperimentalVariables+['Time'])
LatentSpace.reset_index(inplace=True,level='Time')
Conditions_3=LatentSpace.index.unique()

try:
    del(LatentVariable)
except:
    print('Assembling')

for cc in Conditions_3:
    temp=pd.DataFrame(LatentSpace.loc[cc].values,columns=['Time','PCA1','PCA2','PCA3'])
    temp.set_index(['Time'],inplace=True)
    Theta21=compute_angle(temp['PCA1']-temp['PCA1'].loc[initial_timepoint],
                  temp['PCA2']-temp['PCA2'].loc[initial_timepoint],
                  theta_shift,direction='counterclockwise')
    print(Theta21)
    d_21=np.sqrt((temp['PCA2']-temp['PCA2'].loc[initial_timepoint])**2+(temp['PCA1']-temp['PCA1'].loc[initial_timepoint])**2)
    Theta31=compute_angle(temp['PCA1']-temp['PCA1'].loc[initial_timepoint],
                  temp['PCA3']-temp['PCA3'].loc[initial_timepoint],
                  theta_shift,direction='counterclockwise')
    d_31=np.sqrt((temp['PCA3']-temp['PCA3'].loc[initial_timepoint])**2+(temp['PCA1']-temp['PCA1'].loc[initial_timepoint])**2)
    Theta32=compute_angle(temp['PCA2']-temp['PCA2'].loc[initial_timepoint],
                  temp['PCA3']-temp['PCA3'].loc[initial_timepoint],
                  theta_shift,direction='counterclockwise')
    d_32=np.sqrt((temp['PCA3']-temp['PCA3'].loc[initial_timepoint])**2+(temp['PCA2']-temp['PCA2'].loc[initial_timepoint])**2)
    try:
        LatentVariable=pd.concat([LatentVariable,\
                                  pd.DataFrame(np.column_stack([LatentSpace['Time'].loc[cc].values,\
                                                                Theta21,d_21,Theta31,d_31,Theta32,d_32]),\
                  columns=['Time','Theta21','d_21','Theta31','d_31','Theta32','d_32'],\
                index=LatentSpace.loc[cc].index)],axis=0,ignore_index=False)
    except:
        LatentVariable=pd.DataFrame(np.column_stack([LatentSpace['Time'].loc[cc].values,Theta21,d_21,Theta31,d_31,Theta32,d_32]),\
                  columns=['Time','Theta21','d_21','Theta31','d_31','Theta32','d_32'],\
                index=LatentSpace.loc[cc].index)

In [ ]:
# Export latent variables averaged across replicates

save_path = os.path.join(
    save_dir,
    "PCA_AngleDistance_LatentVariable_meanReplicates.xlsx"
)

# Reset index to access columns
df_export = LatentVariable.reset_index()


# Average across replicates
group_cols = ["EffectorTargetRatio", "Donor", "CAR", "Time"]

df_mean = (
    df_export
    .groupby(group_cols, as_index=False)
    .agg({
        "Theta21": "mean",
        "d_21": "mean",
        "Theta31": "mean",
        "d_31": "mean",
        "Theta32": "mean",
        "d_32": "mean",
    })
)


# Export
df_mean.to_excel(save_path, index=False)

print("Saved to:", save_path)

### Parametrization

In [ ]:
#%% Plot the parametrization of the 2D PCA(t) trajectories

Observables = [x for x in LatentVariable.columns if x.find('Time') == -1]

for obs in Observables:

    g = sns.relplot(

        data=LatentVariable.reset_index(),

        x='Time',

        y=obs,

        hue=ExperimentalVariables[0],

        style=ExperimentalVariables[1],

        row=ExperimentalVariables[2],

        col=ExperimentalVariables[3],

        palette='autumn',

        kind='line',

        facet_kws={'sharey': True, 'sharex': True}

    )

    plt.suptitle('Parametrization of PCA(t)\n' + Experiment, fontsize=24)

    g.tight_layout(rect=[0, 0.03, 0.9, 0.9])

    g.savefig(os.path.join(save_dir, f'PCA_Parametrization_{obs}_{Experiment}.pdf'))

In [ ]:
# PCA parametrization: shared setup and data preparation

from matplotlib.lines import Line2D

# Global plotting style
mpl.rcParams["font.family"] = "sans-serif"
mpl.rcParams["font.sans-serif"] = ["Liberation Sans"]
mpl.rcParams["font.size"] = 6
mpl.rcParams["axes.titlesize"] = 6
mpl.rcParams["axes.labelsize"] = 6
mpl.rcParams["xtick.labelsize"] = 4
mpl.rcParams["ytick.labelsize"] = 4
mpl.rcParams["legend.fontsize"] = 6
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42

sns.set_style("white")

SPINE_W = 0.5
TICK_W_MAJOR = 0.5
TICK_LEN_MAJ = 2
TICK_LABEL_PAD = 0.5
EXPORT_DPI = 600

# CAR colors
custom_palette = {
    "Mock": "gray",
    "19_BBz": "red",
    "19_28z": "darkred",
    "22_BBz": "dodgerblue",
    "22_28z": "darkblue",
    "33_BBz": "mediumseagreen",
    "33_28z": "darkgreen",
    "None": "black",
}

# Labels
nice_labels = {
    "Theta21": r"$\theta_{21}$",
    "d_21": r"$d_{21}$",
    "Theta31": r"$\theta_{31}$",
    "d_31": r"$d_{31}$",
    "Theta32": r"$\theta_{32}$",
    "d_32": r"$d_{32}$",
}

# E:T ratio line styles
et_dash = {
    "2": "",
    "1": (2, 1),
    "0.5": (1, 1),
    "0.2": (1, 2),
}

# Variables to plot
observables = [
    "Theta21",
    "d_21",
    "Theta31",
    "d_31",
    "Theta32",
    "d_32",
]

# Prepare latent-variable data
df_plot = LatentVariable.reset_index().copy()

df_plot["EffectorTargetRatio"] = pd.Categorical(
    df_plot["EffectorTargetRatio"].astype(str),
    categories=["2", "1", "0.5", "0.2"],
    ordered=True
)

# Average across replicates while retaining donors
df_mean = (
    df_plot
    .groupby(
        ["CAR", "Donor", "EffectorTargetRatio", "Time"],
        as_index=False,
        observed=True
    )[observables]
    .mean()
)

# Average across donors
df_mean2 = (
    df_mean
    .groupby(
        ["CAR", "EffectorTargetRatio", "Time"],
        as_index=False,
        observed=True
    )[observables]
    .mean()
)

# PCA trajectories averaged across replicates while retaining donors
PCA_mean = (
    PCA_LogAverageOverTime_df
    .groupby(
        ["CAR", "Donor", "EffectorTargetRatio", "Time"],
        as_index=False,
        observed=True
    )[["PCA1", "PCA2"]]
    .mean()
)

# Orders used for legends
car_order = [
    c for c in custom_palette
    if c in df_mean["CAR"].unique()
]

et_order = ["2", "1", "0.5", "0.2"]

car_label_map = {
    "19_BBz": "CD19BBζ",
    "19_28z": "CD1928ζ",
    "22_BBz": "CD22BBζ",
    "22_28z": "CD2228ζ",
    "33_BBz": "CD33BBζ",
    "33_28z": "CD3328ζ",
    "Mock": "Mock",
    "None": "None",
}

In [ ]:
# PCA parametrization plots

# ============================================================
# 1. DONOR-RESOLVED PARAMETRIZATION
# ============================================================

mean_lw = 0.45
mean_alpha = 0.55

for obs in observables:

    fig, ax = plt.subplots(figsize=(2.4, 1.6))

    sns.lineplot(
        data=df_mean,
        x="Time",
        y=obs,
        hue="CAR",
        style="EffectorTargetRatio",
        units="Donor",
        estimator=None,
        dashes=et_dash,
        palette=custom_palette,
        lw=mean_lw,
        alpha=mean_alpha,
        legend=True,
        ax=ax
    )

    for s in ["left", "bottom"]:
        ax.spines[s].set_linewidth(SPINE_W)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax.tick_params(
        axis="both",
        which="major",
        direction="out",
        length=TICK_LEN_MAJ,
        width=TICK_W_MAJOR,
        pad=TICK_LABEL_PAD
    )

    ax.set_xlabel("Time")
    ax.set_ylabel(nice_labels.get(obs, obs))
    ax.set_title("")

    leg = ax.legend(
        frameon=False,
        loc="center left",
        bbox_to_anchor=(1.02, 0.5),
        title="CAR / E:T"
    )

    fig.subplots_adjust(
        left=0.22,
        right=0.80,
        bottom=0.22,
        top=0.98
    )

    save_path = os.path.join(
        save_dir,
        f"PCA_Parametrization_donorResolved_{obs}_{Experiment}.pdf"
    )

    fig.savefig(
        save_path,
        format="pdf",
        dpi=EXPORT_DPI,
        bbox_inches="tight",
        bbox_extra_artists=(leg,)
    )

    plt.show()


# ============================================================
# 2. DONOR + DONOR-AVERAGED PARAMETRIZATION
# ============================================================

donor_lw = 0.30
donor_alpha = 0.20
summary_lw = 0.95

for obs in observables:

    fig, ax = plt.subplots(figsize=(2.4, 1.6))

    # Donor-resolved curves
    sns.lineplot(
        data=df_mean,
        x="Time",
        y=obs,
        hue="CAR",
        style="EffectorTargetRatio",
        units="Donor",
        estimator=None,
        dashes=et_dash,
        palette=custom_palette,
        lw=donor_lw,
        alpha=donor_alpha,
        legend=False,
        ax=ax
    )

    # Donor-averaged curves
    sns.lineplot(
        data=df_mean2,
        x="Time",
        y=obs,
        hue="CAR",
        style="EffectorTargetRatio",
        dashes=et_dash,
        palette=custom_palette,
        lw=summary_lw,
        alpha=1.0,
        legend=True,
        ax=ax
    )

    for s in ["left", "bottom"]:
        ax.spines[s].set_linewidth(SPINE_W)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax.tick_params(
        axis="both",
        which="major",
        direction="out",
        length=TICK_LEN_MAJ,
        width=TICK_W_MAJOR,
        pad=TICK_LABEL_PAD
    )

    ax.set_xlabel("Time")
    ax.set_ylabel(nice_labels.get(obs, obs))
    ax.set_title("")

    ax.legend(
        frameon=False,
        loc="center left",
        bbox_to_anchor=(1.02, 0.5),
        title="CAR / E:T"
    )

    fig.subplots_adjust(
        left=0.22,
        right=0.75,
        bottom=0.22,
        top=0.98
    )

    save_path = os.path.join(
        save_dir,
        f"PCA_Parametrization_donorPlusMean_{obs}_{Experiment}.pdf"
    )

    fig.savefig(
        save_path,
        format="pdf",
        dpi=EXPORT_DPI,
        bbox_inches="tight"
    )

    plt.show()


# ============================================================
# 3. THETA21 + d21 SIDE-BY-SIDE
# ============================================================

fig, axes = plt.subplots(
    1, 2,
    figsize=(3.5, 1.5),
    sharex=True
)

for ax, obs in zip(axes, ["Theta21", "d_21"]):

    sns.lineplot(
        data=df_mean,
        x="Time",
        y=obs,
        hue="CAR",
        hue_order=car_order,
        style="EffectorTargetRatio",
        style_order=et_order,
        units="Donor",
        estimator=None,
        dashes=et_dash,
        palette=custom_palette,
        lw=mean_lw,
        alpha=mean_alpha,
        legend=False,
        ax=ax
    )

    for s in ["left", "bottom"]:
        ax.spines[s].set_linewidth(SPINE_W)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax.tick_params(
        axis="both",
        which="major",
        direction="out",
        length=TICK_LEN_MAJ,
        width=TICK_W_MAJOR,
        pad=TICK_LABEL_PAD
    )

    ax.set_xlabel("Time")
    ax.set_ylabel(nice_labels.get(obs, obs))
    ax.set_title("")

car_handles = [
    Line2D(
        [0], [0],
        color=custom_palette[car],
        lw=1.0,
        label=car_label_map.get(car, car)
    )
    for car in car_order
]

et_handles = [
    Line2D(
        [0], [0],
        color="black",
        lw=1.0,
        linestyle="-" if et_dash[et] == "" else (0, et_dash[et]),
        label=et
    )
    for et in et_order
]

leg_car = fig.legend(
    handles=car_handles,
    title="CAR",
    loc="center left",
    bbox_to_anchor=(0.76, 0.60),
    frameon=False,
    fontsize=6,
    title_fontsize=6,
    handlelength=1.2,
    handletextpad=0.4
)

leg_et = fig.legend(
    handles=et_handles,
    title="E:T ratio",
    loc="center left",
    bbox_to_anchor=(0.96, 0.60),
    frameon=False,
    fontsize=6,
    title_fontsize=6,
    handlelength=1.2,
    handletextpad=0.4
)

fig.subplots_adjust(
    left=0.18,
    right=0.75,
    bottom=0.22,
    top=0.98,
    wspace=0.35
)

save_path = os.path.join(
    save_dir,
    f"PCA_Parametrization_donorResolved_Theta21_d21_{Experiment}.pdf"
)

fig.savefig(
    save_path,
    format="pdf",
    dpi=EXPORT_DPI,
    bbox_inches="tight",
    bbox_extra_artists=(leg_car, leg_et)
)

plt.show()


# ============================================================
# 4. COMBINED PCA2D + THETA21 + d21
# ============================================================

fig, axes = plt.subplots(
    1, 3,
    figsize=(3.75, 1.2),
    gridspec_kw={"width_ratios": [1.2, 1, 1]}
)

# PCA trajectories
sns.lineplot(
    data=PCA_mean,
    x="PCA1",
    y="PCA2",
    hue="CAR",
    style="EffectorTargetRatio",
    units="Donor",
    estimator=None,
    dashes=et_dash,
    palette=custom_palette,
    lw=mean_lw,
    alpha=mean_alpha,
    legend=False,
    ax=axes[0]
)

axes[0].set_xlabel("PCA1")
axes[0].set_ylabel("PCA2")

# Theta21
sns.lineplot(
    data=df_mean,
    x="Time",
    y="Theta21",
    hue="CAR",
    style="EffectorTargetRatio",
    units="Donor",
    estimator=None,
    dashes=et_dash,
    palette=custom_palette,
    lw=mean_lw,
    alpha=mean_alpha,
    legend=False,
    ax=axes[1]
)

axes[1].set_xlabel("Time")
axes[1].set_ylabel(nice_labels["Theta21"])

# d21
sns.lineplot(
    data=df_mean,
    x="Time",
    y="d_21",
    hue="CAR",
    style="EffectorTargetRatio",
    units="Donor",
    estimator=None,
    dashes=et_dash,
    palette=custom_palette,
    lw=mean_lw,
    alpha=mean_alpha,
    legend=False,
    ax=axes[2]
)

axes[2].set_xlabel("Time")
axes[2].set_ylabel(nice_labels["d_21"])

for ax in axes:

    for s in ["left", "bottom"]:
        ax.spines[s].set_linewidth(SPINE_W)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax.tick_params(
        axis="both",
        which="major",
        direction="out",
        length=TICK_LEN_MAJ,
        width=TICK_W_MAJOR,
        pad=TICK_LABEL_PAD
    )

    ax.set_title("")

fig.subplots_adjust(
    left=0.12,
    right=0.98,
    bottom=0.22,
    top=0.95,
    wspace=0.35
)

save_path = os.path.join(
    save_dir,
    f"Combined_PCA_Trajectories_Parametrization_donorAveraged_{Experiment}.pdf"
)

fig.savefig(
    save_path,
    format="pdf",
    dpi=EXPORT_DPI,
    bbox_inches="tight"
)

plt.show()

### Compute median trajectory parameters

In [ ]:
#%% Compute average parametrization  (speed & angle)

try:
    del(AverageLatentVariable)
except:
    print('Assembling')

for cc in Conditions_3:
    temp = LatentVariable.loc[cc].median()
    temp2 = (LatentVariable.loc[cc] / (LatentSpace['Time'].loc[cc].values[:, None])).median()
    try:
        AverageLatentVariable = pd.concat([
            AverageLatentVariable,
            pd.DataFrame(
                np.column_stack([
                    temp['Theta21'], temp2['d_21'],
                    temp['Theta31'], temp2['d_31'],
                    temp['Theta32'], temp2['d_32']
                ]),
                columns=['<Theta21>', '<v_21>', '<Theta31>', '<v_31>', '<Theta32>', '<v_32>']
            )
        ], axis=0, ignore_index=False)
    except:
        AverageLatentVariable = pd.DataFrame(
            np.column_stack([
                temp['Theta21'], temp2['d_21'],
                temp['Theta31'], temp2['d_31'],
                temp['Theta32'], temp2['d_32']
            ]),
            columns=['<Theta21>', '<v_21>', '<Theta31>', '<v_31>', '<Theta32>', '<v_32>']
        )

AverageLatentVariable.index = Conditions_3
AverageLatentVariable.reset_index(inplace=True)

#### Plot median traj parameters

In [ ]:
# ==========================================
# GLOBAL STYLE (Prism-thin)
# ==========================================
mpl.rcParams["font.family"] = "sans-serif"
mpl.rcParams["font.sans-serif"] = ["Liberation Sans"]
mpl.rcParams["font.size"] = 6
mpl.rcParams["axes.titlesize"] = 8
mpl.rcParams["axes.labelsize"] = 6
mpl.rcParams["xtick.labelsize"] = 4
mpl.rcParams["ytick.labelsize"] = 4
mpl.rcParams["legend.fontsize"] = 5
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42

SPINE_W        = 0.5
TICK_W_MAJOR   = 0.5
TICK_LEN_MAJ   = 2
TICK_LABEL_PAD = 0.5

EXPORT_DPI = 600
sns.set_style("white")

# ==========================================
# SAVE LOCATION
# ==========================================
os.makedirs(save_dir, exist_ok=True)

# ==========================================
# USER OPTIONS
# ==========================================
custom_palette = {
    "Mock": "gray",
    "19_BBz": "red",
    "19_28z": "darkred",
    "22_BBz": "dodgerblue",
    "22_28z": "darkblue",
    "33_BBz": "mediumseagreen",
    "33_28z": "darkgreen",
    "None": "black",
}

marker_list = ["o", "s", "D", "^", "v", "P", "X", "*"]

# E:T ratio encoded by point size
size_map = {
    "2": 10,
    "1": 6,
    "0.5": 3,
    "0.2": 1.5,
}

point_alpha   = 0.95
point_edge_lw = 0.2

# ==========================================
# PREPARE DATA
# ==========================================
df_plot = AverageLatentVariable.copy()

possible_donor_cols = ["Donor", "donor", "DONOR", "HealthyDonor", "HD", "Patient"]
donor_col = None
for c in possible_donor_cols:
    if c in df_plot.columns:
        donor_col = c
        break

if donor_col is None:
    raise ValueError(
        "No donor column found. Please set donor_col manually "
        "(for example donor_col = 'Donor')."
    )

print("Using donor column:", donor_col)

df_plot["EffectorTargetRatio"] = (
    pd.to_numeric(df_plot["EffectorTargetRatio"], errors="coerce")
    .map({2.0: "2", 1.0: "1", 0.5: "0.5", 0.2: "0.2"})
)

df_plot["EffectorTargetRatio"] = pd.Categorical(
    df_plot["EffectorTargetRatio"],
    categories=["2", "1", "0.5", "0.2"],
    ordered=True
)

df_plot = df_plot.dropna(subset=["CAR", donor_col, "EffectorTargetRatio"])

# ==========================================
# PARAMETER PAIRS (Angle vs Speed)
# ==========================================
plot_pairs = [
    ("<Theta21>", "<v_21>", r"Angle $\theta_{21}$", r"Speed $v_{21}$", "21"),
    ("<Theta31>", "<v_31>", r"Angle $\theta_{31}$", r"Speed $v_{31}$", "31"),
    ("<Theta32>", "<v_32>", r"Angle $\theta_{32}$", r"Speed $v_{32}$", "32"),
]

# ==========================================
# PLOT
# 1 point = mean of 3 replicates for one CAR x E:T x donor
# color = CAR
# shape = donor
# size  = E:T ratio
# ==========================================
for xcol, ycol, xlabel, ylabel, suffix in plot_pairs:

    df_donor = (
        df_plot
        .groupby(
            ["CAR", "EffectorTargetRatio", donor_col],
            as_index=False,
            observed=False
        )[[xcol, ycol]]
        .mean()
        .sort_values(["CAR", donor_col, "EffectorTargetRatio"])
    )

    print(f"\nPanel {suffix}: donor-level counts")
    print(
        df_donor
        .groupby(["CAR", "EffectorTargetRatio"], observed=False)
        .size()
    )

    donor_list = list(df_donor[donor_col].drop_duplicates())

    marker_map = {
        donor: marker_list[i % len(marker_list)]
        for i, donor in enumerate(donor_list)
    }

    fig, ax = plt.subplots(figsize=(2.2, 1.7))

    # Donor points only
    for donor in donor_list:
        sub = df_donor[df_donor[donor_col] == donor].copy()

        for et in ["2", "1", "0.5", "0.2"]:
            sub_et = sub[sub["EffectorTargetRatio"] == et]

            if len(sub_et) == 0:
                continue

            ax.scatter(
                sub_et[xcol].values,
                sub_et[ycol].values,
                c=sub_et["CAR"].map(custom_palette).values,
                s=size_map[et],
                alpha=point_alpha,
                edgecolor="black",
                linewidth=point_edge_lw,
                marker=marker_map[donor],
                zorder=2
            )

    # Aesthetics
    for s in ["left", "bottom"]:
        ax.spines[s].set_linewidth(SPINE_W)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax.tick_params(
        axis="both",
        which="major",
        direction="out",
        length=TICK_LEN_MAJ,
        width=TICK_W_MAJOR,
        pad=TICK_LABEL_PAD
    )

    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title("")

    fig.subplots_adjust(
        left=0.25,
        right=0.95,
        bottom=0.25,
        top=0.95
    )

    save_path = os.path.join(
        save_dir,
        f"PCA_AngleVsSpeed_DonorShape_ETsize_{suffix}_{Experiment}.pdf"
    )

    fig.savefig(
        save_path,
        format="pdf",
        dpi=EXPORT_DPI
    )

    print("Saved:", save_path)
    plt.show()

#### BarPlot theta21-angle values

In [ ]:
# ==========================================
# BARPLOT : median Theta21 per CAR
# with donor/E:T points
# ==========================================


# ==========================================
# PARAMETERS
# ==========================================

xcol = "<Theta21>"
ylabel = r"Angle $\theta_{21}$"

# CAR order — Mock and None excluded
car_order = [
    "19_28z", "19_BBz",
    "22_28z", "22_BBz",
    "33_28z", "33_BBz", "Mock", "None"
]

# Custom y-axis limits
# Keep None for automatic scaling
ymin = 0.35
ymax = 0.55


# ==========================================
# DONOR-LEVEL DATA
# 1 point = mean of replicates for one CAR x E:T x donor
# ==========================================

df_theta = (
    df_plot
    .groupby(
        ["CAR", "EffectorTargetRatio", donor_col],
        as_index=False,
        observed=False
    )[[xcol]]
    .mean()
)

# Explicitly remove Mock / None / anything not in car_order
df_theta = df_theta[df_theta["CAR"].isin(car_order)].copy()

# Keep only CARs actually present in the dataframe
car_order = [
    car for car in car_order
    if car in df_theta["CAR"].unique()
]

# ==========================================
# MEDIAN PER CAR
# ==========================================

bar_df = (
    df_theta
    .groupby("CAR", observed=False)[xcol]
    .median()
    .reindex(car_order)
    .reset_index()
)

# ==========================================
# DONOR MARKERS
# ==========================================

donor_list = list(df_theta[donor_col].drop_duplicates())

marker_map = {
    donor: marker_list[i % len(marker_list)]
    for i, donor in enumerate(donor_list)
}

# ==========================================
# FIGURE
# ==========================================

fig, ax = plt.subplots(figsize=(2.0, 1.8))

# Bars = median Theta21 per CAR
ax.bar(
    np.arange(len(bar_df)),
    bar_df[xcol].values,
    color=[custom_palette[c] for c in bar_df["CAR"]],
    edgecolor="black",
    linewidth=0.4,
    width=0.72,
    alpha=0.85,
    zorder=1
)

# ==========================================
# OVERLAY POINTS
# color = CAR
# shape = donor
# size = E:T ratio
# ==========================================

rng = np.random.default_rng(0)

for i, car in enumerate(car_order):

    sub_car = df_theta[df_theta["CAR"] == car]

    for donor in donor_list:

        sub_donor = sub_car[sub_car[donor_col] == donor]

        for et in ["2", "1", "0.5", "0.2"]:

            sub_et = sub_donor[
                sub_donor["EffectorTargetRatio"] == et
            ]

            if len(sub_et) == 0:
                continue

            jitter = rng.normal(0, 0.06, len(sub_et))

            ax.scatter(
                np.full(len(sub_et), i) + jitter,
                sub_et[xcol].values,
                c=custom_palette[car],
                s=size_map[et],
                alpha=point_alpha,
                edgecolor="black",
                linewidth=point_edge_lw,
                marker=marker_map[donor],
                zorder=3
            )

# ==========================================
# CUSTOM Y-AXIS
# ==========================================

if ymin is not None or ymax is not None:
    ax.set_ylim(ymin, ymax)

# ==========================================
# AESTHETICS
# ==========================================

for s in ["left", "bottom"]:
    ax.spines[s].set_linewidth(SPINE_W)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.tick_params(
    axis="both",
    which="major",
    direction="out",
    length=TICK_LEN_MAJ,
    width=TICK_W_MAJOR,
    pad=TICK_LABEL_PAD
)

ax.set_xticks(np.arange(len(car_order)))
ax.set_xticklabels(
    car_order,
    rotation=45,
    ha="right"
)

ax.set_ylabel(ylabel)
ax.set_xlabel("")

fig.subplots_adjust(
    left=0.24,
    right=0.98,
    bottom=0.33,
    top=0.98
)

plt.show()

In [ ]:
# ==========================================
# BARPLOT : median Theta21 per CAR
# Stats: paired t-tests across targets within each costim
# Pairing = donor + E:T ratio
# ==========================================

# ==========================================
# PRISM-THIN STYLE
# ==========================================

mpl.rcParams["font.family"] = "sans-serif"
mpl.rcParams["font.sans-serif"] = ["Liberation Sans"]

mpl.rcParams["font.size"] = 6
mpl.rcParams["axes.titlesize"] = 4
mpl.rcParams["axes.labelsize"] = 8
mpl.rcParams["xtick.labelsize"] = 5
mpl.rcParams["ytick.labelsize"] = 5
mpl.rcParams["legend.fontsize"] = 6

mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42

SPINE_W        = 0.5
BAR_EDGE_W     = 0.25
TICK_W_MAJOR   = 0.5
TICK_LEN_MAJ   = 2
TICK_LABEL_PAD = 0.5
EXPORT_DPI     = 600

sns.set_style("white")

# ==========================================
# PARAMETERS
# ==========================================

xcol = "<Theta21>"

ymin = 0.35
ymax = 0.55

fig_width  = 2
fig_height = 2

bar_alpha = 0.45

car_order = [
    "19_28z", "19_BBz",
    "22_28z", "22_BBz",
    "33_28z", "33_BBz",
    "Mock"
]

sig_pairs = [
    ("19_28z", "22_28z"),
    ("19_28z", "33_28z"),
    ("19_BBz", "22_BBz"),
    ("19_BBz", "33_BBz"),
    ("22_28z", "33_28z"),
    ("22_BBz", "33_BBz"),
]

# ==========================================
# DONOR-LEVEL DATA
# ==========================================

df_theta = (
    df_plot
    .groupby(
        ["CAR", "EffectorTargetRatio", donor_col],
        as_index=False,
        observed=False
    )[[xcol]]
    .mean()
)

df_theta = df_theta[df_theta["CAR"].isin(car_order)].copy()

car_order = [
    c for c in car_order
    if c in df_theta["CAR"].unique()
]

# ==========================================
# MEDIAN PER CAR
# ==========================================

bar_df = (
    df_theta
    .groupby("CAR", observed=False)[xcol]
    .median()
    .reindex(car_order)
    .reset_index()
)

# ==========================================
# PAIRED STATS
# ==========================================

def p_to_star(p):
    if np.isnan(p):
        return "ns"
    elif p < 1e-4:
        return "****"
    elif p < 1e-3:
        return "***"
    elif p < 1e-2:
        return "**"
    elif p < 0.05:
        return "*"
    else:
        return "ns"

paired_table = (
    df_theta
    .pivot_table(
        index=[donor_col, "EffectorTargetRatio"],
        columns="CAR",
        values=xcol
    )
)

stats_rows = []

for car1, car2 in sig_pairs:

    if car1 not in paired_table.columns or car2 not in paired_table.columns:
        continue

    sub = paired_table[[car1, car2]].dropna()

    if len(sub) < 2:
        tstat = np.nan
        pval = np.nan
    else:
        tstat, pval = stats.ttest_rel(sub[car1], sub[car2])

    stats_rows.append({
        "CAR1": car1,
        "CAR2": car2,
        "n_pairs": len(sub),
        "tstat": tstat,
        "pval": pval,
        "stars": p_to_star(pval)
    })

stats_results = pd.DataFrame(stats_rows)
print(stats_results.sort_values("pval"))

# ==========================================
# DONOR MARKERS
# ==========================================

donor_list = list(df_theta[donor_col].drop_duplicates())

marker_map = {
    donor: marker_list[i % len(marker_list)]
    for i, donor in enumerate(donor_list)
}

# ==========================================
# FIGURE
# ==========================================

fig, ax = plt.subplots(figsize=(fig_width, fig_height))

# ==========================================
# BARS
# ==========================================

bar_colors = [
    to_rgba(custom_palette[c], alpha=bar_alpha)
    for c in bar_df["CAR"]
]

ax.bar(
    np.arange(len(bar_df)),
    bar_df[xcol].values,
    color=bar_colors,
    edgecolor="black",
    linewidth=BAR_EDGE_W,
    width=0.72,
    zorder=1
)

# ==========================================
# OVERLAY POINTS
# ==========================================

rng = np.random.default_rng(0)

for i, car in enumerate(car_order):

    sub_car = df_theta[df_theta["CAR"] == car]

    for donor in donor_list:

        sub_donor = sub_car[sub_car[donor_col] == donor]

        for et in ["2", "1", "0.5", "0.2"]:

            sub_et = sub_donor[
                sub_donor["EffectorTargetRatio"] == et
            ]

            if len(sub_et) == 0:
                continue

            jitter = rng.normal(0, 0.06, len(sub_et))

            ax.scatter(
                np.full(len(sub_et), i) + jitter,
                sub_et[xcol].values,
                c=custom_palette[car],
                s=size_map[et],
                alpha=point_alpha,
                edgecolor="black",
                linewidth=point_edge_lw,
                marker=marker_map[donor],
                zorder=3
            )

# ==========================================
# SIGNIFICANCE BRACKETS
# ==========================================

xpos = {car: i for i, car in enumerate(car_order)}

data_ymax = df_theta[xcol].max()
data_ymin = df_theta[xcol].min()
yrange = data_ymax - data_ymin

line_height = 0.02 * yrange
text_offset = 0.01 * yrange
step = 0.08 * yrange

bracket_specs = [
    ("22_BBz", "33_BBz"),
    ("22_28z", "33_28z"),
    ("19_BBz", "22_BBz"),
    ("19_BBz", "33_BBz"),
    ("19_28z", "22_28z"),
    ("19_28z", "33_28z"),
]

for i, (car1, car2) in enumerate(bracket_specs):

    row = stats_results[
        (stats_results["CAR1"] == car1)
        &
        (stats_results["CAR2"] == car2)
    ]

    if len(row) == 0:
        continue

    stars = row["stars"].iloc[0]

    x1 = xpos[car1]
    x2 = xpos[car2]

    bracket_base = 0.51
    bracket_step = 0.016

    y = bracket_base + bracket_step * i

    ax.plot(
        [x1, x1, x2, x2],
        [y, y + line_height, y + line_height, y],
        color="black",
        linewidth=0.45,
        clip_on=False
    )

    ax.text(
        (x1 + x2) / 2,
        y + line_height + text_offset,
        stars,
        ha="center",
        va="bottom",
        fontsize=5,
        clip_on=False
    )

# ==========================================
# AXES
# ==========================================

ax.set_ylabel(
    "Cytokine dynamics\n"
    r"signature $(\theta_{21})$"
)

bottom = (
    data_ymin - 0.05 * yrange
    if ymin is None else ymin
)

top = (
    data_ymax + step * (len(bracket_specs) + 1)
    if ymax is None else ymax
)

ax.set_ylim(bottom, top)
ax.yaxis.set_major_locator(MaxNLocator(nbins=4))

# ==========================================
# AESTHETICS
# ==========================================

for s in ["left", "bottom"]:
    ax.spines[s].set_linewidth(SPINE_W)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.tick_params(
    axis="both",
    which="major",
    direction="out",
    length=TICK_LEN_MAJ,
    width=TICK_W_MAJOR,
    pad=TICK_LABEL_PAD
)

ax.set_xticks(np.arange(len(car_order)))
ax.set_xticklabels([])
ax.tick_params(axis="x", length=0)

ax.set_xlabel("")

fig.subplots_adjust(
    left=0.30,
    right=0.98,
    bottom=0.10,
    top=0.80
)

# ==========================================
# SAVE
# ==========================================

save_path = os.path.join(
    save_dir,
    f"Theta21_Barplot_Raw_{Experiment}.pdf"
)

plt.savefig(
    save_path,
    dpi=EXPORT_DPI
)

print("Saved:", save_path)

plt.show()

#### Impact of costimulatory domain

In [ ]:
#%% Compute Δθ21 as % of CAR-only range


angleDf = AverageLatentVariable.copy()

angleDf["Costim"] = angleDf["CAR"].apply(lambda x: "None" if "_" not in x else x.split("_")[1])
angleDf["scFv"]   = angleDf["CAR"].apply(lambda x: "None" if "_" not in x else x.split("_")[0])

angleDf = angleDf.query("Costim != 'None'").copy()

pairedDf = (
    angleDf
    .pivot(
        index=["EffectorTargetRatio", "Donor", "Replicate", "scFv"],
        columns="Costim",
        values="<Theta21>"
    )
    .dropna(subset=["28z", "BBz"])
)

pairedDf.columns.name = None

theta_range_car = (
    AverageLatentVariable.query("CAR != 'Mock' and CAR != 'None'")["<Theta21>"].max()
    - AverageLatentVariable.query("CAR != 'Mock' and CAR != 'None'")["<Theta21>"].min()
)

pairedDf["Delta_percent_CAR_range"] = (
    (pairedDf["28z"] - pairedDf["BBz"]) / theta_range_car
) * 100

summaryDf_delta_car_range = (
    pairedDf
    .reset_index()
    .groupby(["EffectorTargetRatio", "scFv"])
    .agg(
        mean_delta=("Delta_percent_CAR_range", "mean"),
        sem_delta=("Delta_percent_CAR_range", lambda x: stats.sem(x, nan_policy="omit")),
        n=("Delta_percent_CAR_range", "count")
    )
    .reset_index()
)

display(summaryDf_delta_car_range)

In [ ]:
#%% Plot Δθ21 (% CAR range) — FINAL with visible ticks + clean 2-layer x-axis

# ==========================================
# STYLE
# ==========================================
mpl.rcParams["font.family"] = "DejaVu Sans"
mpl.rcParams["font.size"] = 6
mpl.rcParams["axes.labelsize"] = 6
mpl.rcParams["xtick.labelsize"] = 5
mpl.rcParams["ytick.labelsize"] = 5
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42

SPINE_W = 0.6
TICK_W  = 0.6
TICK_L  = 2.5
EXPORT_DPI = 600

sns.set_style("white")

# ==========================================
# SAVE LOCATION
# ==========================================
os.makedirs(save_dir, exist_ok=True)

# ==========================================
# COLORS
# ==========================================
target_palette = {
    "19": "darkred",
    "22": "darkblue",
    "33": "darkgreen",
}

# ==========================================
# DATA
# ==========================================
plotDf = summaryDf_delta_car_range.copy()
plotDf["EffectorTargetRatio"] = plotDf["EffectorTargetRatio"].astype(str)
plotDf["scFv"] = plotDf["scFv"].astype(str)

et_order = ["2", "1", "0.5", "0.2"]
scfv_order = ["19", "22", "33"]

plotDf["ET_order"] = plotDf["EffectorTargetRatio"].map(
    {v: i for i, v in enumerate(et_order)}
)
plotDf["scFv_order"] = plotDf["scFv"].map(
    {v: i for i, v in enumerate(scfv_order)}
)

plotDf = (
    plotDf
    .sort_values(["scFv_order", "ET_order"])
    .reset_index(drop=True)
)

plotDf["x"] = np.arange(len(plotDf))

# ==========================================
# STATS
# ==========================================
def giveStar(p):
    if np.isnan(p):
        return "ns"
    elif p < 1e-4:
        return "****"
    elif p < 1e-3:
        return "***"
    elif p < 1e-2:
        return "**"
    elif p < 0.05:
        return "*"
    else:
        return "ns"


star_map = {}

for scfv in scfv_order:
    for et in et_order:

        vals = pairedDf.loc[
            (
                pairedDf.index
                .get_level_values("EffectorTargetRatio")
                .astype(str) == et
            )
            &
            (
                pairedDf.index
                .get_level_values("scFv")
                .astype(str) == scfv
            ),
            "Delta_percent_CAR_range"
        ].dropna().values

        pval = (
            stats.ttest_1samp(vals, 0)[1]
            if len(vals) > 1 else np.nan
        )

        star_map[(et, scfv)] = giveStar(pval)

plotDf["star"] = [
    star_map[(row["EffectorTargetRatio"], row["scFv"])]
    for _, row in plotDf.iterrows()
]

# ==========================================
# PLOT
# ==========================================
fig, ax = plt.subplots(figsize=(2.5, 1.9))

ax.bar(
    plotDf["x"],
    plotDf["mean_delta"],
    yerr=plotDf["sem_delta"],
    color=[target_palette[x] for x in plotDf["scFv"]],
    edgecolor="black",
    linewidth=SPINE_W,
    alpha=0.8,
    capsize=2,
    error_kw=dict(
        linewidth=SPINE_W * 0.5,
        capthick=SPINE_W * 0.5
    ),
)

ax.axhline(
    0,
    color="black",
    linewidth=SPINE_W
)

# ==========================================
# STARS
# ==========================================
y_top = np.nanmax(
    plotDf["mean_delta"] + plotDf["sem_delta"]
)

offset = 0.05 * y_top

for _, row in plotDf.iterrows():

    if row["star"] != "ns":
        ax.text(
            row["x"],
            row["mean_delta"] + row["sem_delta"] + offset,
            row["star"],
            ha="center",
            fontsize=6
        )

# ==========================================
# X-AXIS (2 layers)
# ==========================================
ax.set_xticks(plotDf["x"])
ax.set_xticklabels(plotDf["EffectorTargetRatio"])

# --- CUSTOM POSITIONS ---
et_y = -0.08
target_y = -0.18
left_x = -1

ax.text(
    left_x,
    et_y,
    "E:T",
    transform=ax.get_xaxis_transform(),
    ha="right",
    va="center",
    fontsize=6
)

ax.text(
    left_x,
    target_y,
    "Target",
    transform=ax.get_xaxis_transform(),
    ha="right",
    va="center",
    fontsize=6
)

for i, target in enumerate(scfv_order):

    center = i * len(et_order) + 1.5

    ax.text(
        center,
        target_y,
        f"CD{target}",
        transform=ax.get_xaxis_transform(),
        ha="center",
        va="center",
        fontsize=6
    )

for i in range(1, len(scfv_order)):

    xpos = i * len(et_order) - 0.5

    ax.plot(
        [xpos, xpos],
        [0, target_y - 0.05],
        transform=ax.get_xaxis_transform(),
        color="black",
        linewidth=0.6,
        clip_on=False
    )

# ==========================================
# Y-AXIS
# ==========================================
ax.set_ylabel(
    r"$\theta_{21}$ var. (%) "
    r"$\left[\frac{\Delta\theta_{21}^{28\zeta-4\mathrm{-}1\mathrm{BB}\zeta}}"
    r"{\Delta\theta_{21}^{\max-\min}}\times100\right]$"
)

# ==========================================
# FORCE TICKS VISIBLE
# ==========================================
ax.tick_params(
    axis="both",
    which="major",
    direction="out",
    length=TICK_L,
    width=TICK_W,
    colors="black",
    bottom=True,
    left=True,
    labelbottom=True,
    labelleft=True,
    pad=0.5
)

for s in ["left", "bottom"]:
    ax.spines[s].set_linewidth(SPINE_W)
    ax.spines[s].set_color("black")

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# ==========================================
# LIMITS / LAYOUT
# ==========================================
y_bot = np.nanmin(
    plotDf["mean_delta"] - plotDf["sem_delta"]
)

ax.set_ylim(
    y_bot * 1.2,
    y_top * 1.2
)

fig.subplots_adjust(
    left=0.34,
    right=0.95,
    bottom=0.35,
    top=0.95
)

# ==========================================
# SAVE
# ==========================================
save_path = os.path.join(
    save_dir,
    f"DeltaTheta21_FINAL_visibleTicks_noZeroLine_{Experiment}.pdf"
)

fig.savefig(
    save_path,
    dpi=EXPORT_DPI
)

print("Saved:", save_path)

plt.show()

# 6/ Total log-cytokine integrals
## ∫(t₀ → t_final)

## UMAP

In [ ]:
# UMAP of global cytokine secretion (log-cytokine integrals)

# ==========================================
# STYLE
# ==========================================
mpl.rcParams["font.family"] = "sans-serif"
mpl.rcParams["font.sans-serif"] = ["Liberation Sans"]
mpl.rcParams["font.size"] = 6
mpl.rcParams["axes.labelsize"] = 6
mpl.rcParams["xtick.labelsize"] = 4
mpl.rcParams["ytick.labelsize"] = 4
mpl.rcParams["legend.fontsize"] = 5
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42

sns.set_style("white")

# ==========================================
# SAVE
# ==========================================
save_path = os.path.join(
    save_dir,
    "UMAP2D_final_logIntegral_allRep_points.pdf"
)

# ==========================================
# COLORS / MARKERS
# ==========================================
custom_palette = {
    "Mock": "gray",
    "19_BBz": "red",
    "19_28z": "darkred",
    "22_BBz": "dodgerblue",
    "22_28z": "darkblue",
    "33_BBz": "mediumseagreen",
    "33_28z": "darkgreen",
    "None": "black",
}

donor_markers = {
    "A": "o",
    "B": "s",
    "C": "^",
    "D": "D",
}

# ==========================================
# CUSTOMIZABLE PARAMETERS
# ==========================================
fig_width = 2
fig_height = 1.75

point_size_min = 1
point_size_max = 8
point_alpha = 0.80
point_edge_lw = 0.10

umap_n_neighbors = 15
umap_metric = "euclidean"
umap_min_dist = 0.15
umap_random_state = 45

xlim = None
ylim = None

# ==========================================
# DATA
# ==========================================
df_plot = LogIntegral_df.copy()

if isinstance(df_plot.columns, pd.MultiIndex):
    df_plot.columns = df_plot.columns.get_level_values(-1)

df_plot = df_plot.reset_index()
df_plot["Time"] = pd.to_numeric(df_plot["Time"], errors="coerce")

df_plot = (
    df_plot
    .sort_values("Time")
    .groupby(
        ["EffectorTargetRatio", "Donor", "CAR", "Replicate"],
        as_index=False
    )
    .last()
)

meta_cols = [
    "EffectorTargetRatio",
    "Donor",
    "CAR",
    "Replicate",
    "Time"
]

cytokine_cols = [
    c for c in df_plot.columns
    if c not in meta_cols
]

df_plot["EffectorTargetRatio"] = pd.Categorical(
    df_plot["EffectorTargetRatio"].astype(str),
    categories=["2", "1", "0.5", "0.2"],
    ordered=True
)

# ==========================================
# UMAP
# ==========================================
X = df_plot[cytokine_cols].values

reducer = umap.UMAP(
    n_neighbors=umap_n_neighbors,
    metric=umap_metric,
    min_dist=umap_min_dist,
    random_state=umap_random_state
)

df_plot[["UMAP1", "UMAP2"]] = reducer.fit_transform(X)

# ==========================================
# PLOT
# ==========================================
fig, ax = plt.subplots(figsize=(fig_width, fig_height))

sns.scatterplot(
    data=df_plot,
    x="UMAP1",
    y="UMAP2",
    hue="CAR",
    style="Donor",
    size="EffectorTargetRatio",
    palette=custom_palette,
    markers=donor_markers,
    sizes=(point_size_min, point_size_max),
    alpha=point_alpha,
    linewidth=point_edge_lw,
    edgecolor="black",
    legend=False,
    ax=ax
)

for s in ["left", "bottom"]:
    ax.spines[s].set_linewidth(0.5)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.tick_params(
    axis="both",
    direction="out",
    length=2,
    width=0.5,
    pad=0.5
)

ax.set_xlabel("UMAP 1")
ax.set_ylabel("UMAP 2")

if xlim is not None:
    ax.set_xlim(xlim)

if ylim is not None:
    ax.set_ylim(ylim)

fig.subplots_adjust(
    left=0.16,
    right=0.98,
    bottom=0.22,
    top=0.96
)

fig.savefig(
    save_path,
    format="pdf",
    dpi=600,
    bbox_inches="tight",
    pad_inches=0.02
)

plt.show()